In [33]:
!pip install webdriver-manager

  Obtaining dependency information for webdriver-manager from https://files.pythonhosted.org/packages/b5/b5/3bd0b038d80950ec13e6a2c8d03ed8354867dc60064b172f2f4ffac8afbe/webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata


newer 

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.action_chains import ActionChains
import time
import pyautogui
import datetime

# -------------------------------
# Helper: Get formatted date string
# -------------------------------
def get_date_two_weeks_ahead() -> str:
    """Return formatted date string exactly 2 weeks from today."""
    target_date = datetime.date.today() + datetime.timedelta(weeks=2)
    try:
        return target_date.strftime("%A, %B %-d, %Y")  # Linux/Mac
    except:
        return target_date.strftime("%A, %B %#d, %Y")  # Windows fallback


# -------------------------------
# Initialize driver
# -------------------------------
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)
actions = ActionChains(driver)

def safe_click(element):
    """Try multiple methods to click an element"""
    try:
        element.click()  # Standard click
    except:
        try:
            driver.execute_script("arguments[0].click();", element)  # JS click
        except:
            try:
                x, y = element.location['x'], element.location['y']
                pyautogui.click(x + 10, y + 10)
                time.sleep(0.5)
            except Exception as e:
                print(f"All click methods failed: {str(e)}")
                raise

try:
    # -------------------------------
    # Load page and add cookies
    # -------------------------------
    driver.get("https://libcal.library.gatech.edu/reserve/study-rooms")
    cookies = [
        {"name": "lc_ea_po", "value": "00155f22c32d868e39557048690461f7a800cbf0614bb20aa7f4fd1cf9227fe8040a7c6089b9467a534df74b273e761646e4490a9898147a90b09201b74d4a935002112d830dfdc75b0ff2f4fe124e532a359385753272c266a08d54e56fed19113e78b7e0193d98c19ff4d42a00bc34de6506099ea4b2c77e890524e1c0c066472f8947c40af0bc7b53e2a350b91132d2fa52a94d87b61ed09c80df07b48eb376c9a9e2599f2692d80c21aca8b0e"},
        {"name": "lc_ebcart", "value": "001a479ddf130eac886a058a332ca3a15f14bcba037fa964bb4c51d1033d57e8d71058f4604b79d2e0eafd506dc1caa65e2bdea430a511f22762bd541c064ba25e8d911f0f18bd1bcb01ccc29e0f3efef593df3e705bc975bb8b077226ab9516bdf3ab79e33604b7f6ecda7f891f96913690be7730bca86efd494453500b60e368e79454afb7bf28c0200e7bb1b35e427d9ba1b765a6465eda7a36b419c25b2c1de7c9b92e13b9f990888e66b29feb474774e02df99ac204e686ba5e707b1e03f4a91387341511f1184a7d58589f0"}
    ]
    for cookie in cookies:
        driver.add_cookie(cookie)
    driver.refresh()

    # -------------------------------
    # Target: exactly 2 weeks ahead
    # -------------------------------
    target_time = "5:00pm"
    target_date_str = get_date_two_weeks_ahead()

    while True:
        toolbar = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "fc-toolbar-title")))
        if target_date_str.split(",")[0] in toolbar.text:  # check weekday matches
            break
        next_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".btn-group .fc-next-button")))
        safe_click(next_btn)
        time.sleep(1)

    # -------------------------------
    # Select time slot
    # -------------------------------
    slot_xpath = f"//a[contains(@class,'fc-timeline-event') and contains(@title,'{target_time} {target_date_str} - Clough 342 - Available')]"
    slot = wait.until(EC.element_to_be_clickable((By.XPATH, slot_xpath)))
    driver.execute_script("arguments[0].scrollIntoView(true);", slot)
    time.sleep(0.5)
    safe_click(slot)

    # -------------------------------
    # Select maximum end time
    # -------------------------------
    end_dropdown = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'select.b-end-date')))
    select = Select(end_dropdown)
    select.select_by_index(len(select.options) - 1)
    time.sleep(1)

    # -------------------------------
    # Submit
    # -------------------------------
    submit_btn = wait.until(EC.element_to_be_clickable((By.ID, "submit_times")))
    driver.execute_script("arguments[0].scrollIntoView(true);", submit_btn)
    safe_click(submit_btn)
    print("Submit button clicked - script complete")

except Exception as e:
    print(f"Error occurred: {str(e)}")
    driver.save_screenshot("error.png")

finally:
    input("Press Enter to close browser...")
    driver.quit()


Error occurred: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff67bd8aa55
	0x7ff67bae8630
	0x7ff67b87d75d
	0x7ff67b8d862e
	0x7ff67b8d893c
	0x7ff67b928d87
	0x7ff67b92591c
	0x7ff67b8c9098
	0x7ff67b8c9f83
	0x7ff67bdb7810
	0x7ff67bdb1afd
	0x7ff67bdd2c1a
	0x7ff67bb03345
	0x7ff67bb0b81c
	0x7ff67baf1924
	0x7ff67baf1ad6
	0x7ff67bad7e47
	0x7ffb4785e8d7
	0x7ffb4966c48c

